In [7]:
import torch
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import io
import requests
from IPython.display import display
from diffusers import StableDiffusionXLImg2ImgPipeline


def load_image(image_path_or_url):
    """Load image from file path or URL"""
    try:
        if image_path_or_url.startswith(('http://', 'https://')):
            response = requests.get(image_path_or_url)
            image = Image.open(io.BytesIO(response.content))
        else:
            image = Image.open(image_path_or_url)
        return image
    except Exception as e:
        print(f"Error loading image: {e}")
        return None

def preprocess_image(image, max_size=512):
    """Resize and preprocess image for the model"""
    # Calculate new dimensions while maintaining aspect ratio
    width, height = image.size
    if width > height:
        new_width = max_size
        new_height = int(height * (max_size / width))
    else:
        new_height = max_size
        new_width = int(width * (max_size / height))
    
    # Resize image
    image = image.resize((new_width, new_height), Image.LANCZOS)
    
    # Convert to RGB if needed
    if image.mode != "RGB":
        image = image.convert("RGB")
        
    return image

def generate_avatar(input_image_path, prompt, strength=0.75, guidance_scale=7.5):
    """Generate avatar based on input image and prompt"""
    # Load and preprocess input image
    input_image = load_image(input_image_path)
    if input_image is None:
        return None
    
    input_image = preprocess_image(input_image)
    
    # Device detection
    print("=" * 50)
    print(f"CUDA available: {torch.cuda.is_available()}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    if device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("=" * 50)
    
    # Load Stable Diffusion XL pipeline
    model_id = "stabilityai/stable-diffusion-xl-base-1.0"
    pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        variant="fp16",
        use_safetensors=True
    ).to(device)
    
    # Enhanced prompt for avatar generation
    enhanced_prompt = f"profile picture avatar of person, {prompt}, solid background"
    
    # Generate avatar
    result = pipe(
        prompt=enhanced_prompt,
        image=input_image,
        strength=strength,
        guidance_scale=guidance_scale
    ).images[0]
    
    return result

def display_comparison(original_image, generated_avatar):
    """Display original image and generated avatar side by side"""
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.imshow(original_image)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(generated_avatar)
    plt.title('Generated Avatar')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

def save_avatar(avatar, output_path="generated_avatar.png"):
    """Save the generated avatar to a file"""
    avatar.save(output_path)
    print(f"Avatar saved to {output_path}")

In [8]:
def create_avatar_from_image(image_path, prompt, output_path=None, show_comparison=True):
    """
    Generate an avatar from an input image and text prompt
    
    Parameters:
    - image_path: Path or URL to input image
    - prompt: Text description for avatar style (e.g., "anime style", "digital art", "3D cartoon")
    - output_path: Where to save the output (optional)
    - show_comparison: Whether to display before/after comparison
    """
    # Load input image for display
    input_image = load_image(image_path)
    if input_image is None:
        return
    
    print("Generating avatar... This may take a minute.")
    avatar = generate_avatar(image_path, prompt)
    
    if show_comparison:
        display_comparison(input_image, avatar)
    
    if output_path:
        save_avatar(avatar, output_path)
    
    return avatar

In [ ]:
# Example
input_image_path = "input_image.jpg"  # Replace with your image path
text_prompt = "cartoon"  # Customize this prompt

avatar = create_avatar_from_image(
    image_path=input_image_path,
    prompt=text_prompt,
    output_path="my_new_avatar.png"
)